# Анализ траекторий МД: коэффициенты диффузии и радиальные функции распределения (RDF), учет периодических граничных условий

**Цель:** увидеть, что пересечение границы периодической ячейки меняет представление координат, но не физическую траекторию. Мы создадим небольшой куб воды, покажем исходные координаты из OpenMM, wrapped- и unwrapped-представления, соседние образы, а затем сравним правильный и ошибочный расчёты MSD (для коэффициентов диффузии) и RDF.

Запускайте ячейки сверху вниз в Google Colab. GPU желателен, но предусмотрен CPU. Установка требует доступа к интернету; исходные структуры скачивать не нужно. Расчёт включает 20 пс NVT-релаксации и 60 пс NVT-наблюдения, затем 2 пс переключения на NVE и 40 пс NVE-наблюдения с частой записью скоростей. Время выполнения зависит от платформы; это учебная траектория, а не измерение коэффициента диффузии воды с высокой точностью.

### 1. Как подготовить систему в OpenMM

1. **Топология (Topology)** хранит химическую структуру: цепи → остатки HOH → атомы O, H1, H2 и связи O–H. Координаты хранятся отдельно; наличие связи в Topology ещё не задаёт численного потенциала.
2. **Силовое поле (ForceField, 'tip3p.xml')** загружает шаблон воды и параметры. Здесь используется стандартная трёхточечная модель TIP3P, а не CHARMM-модификация TIP3P.
3. **Modeller.addSolvent** заполняет пустую топологию уравновешенными молекулами воды из пакета OpenMM, обрезая/размножая заготовку под заданный размер бокса. Это не случайная расстановка атомов и не построение молекулы по одному атому. Число молекул зависит от размера ячейки и версии заготовки.
4. **ForceField.createSystem** превращает топологию в численную модель: массы, заряды, взаимодействия и ограничения. Задаём PME для периодической электростатики, cutoff 0.8 нм для прямой части и LJ, жёсткую геометрию воды и шаг 2 фс.
5. **Integrator** задаёт динамику и термостат. Используем LangevinMiddle, 300 K, трение 1 пс$^{-1}$. **Simulation/Context** объединяет топологию, System, интегратор и вычислительную платформу.
6. Назначаем координаты, минимизируем энергию, задаём начальные скорости из распределения Максвелла, релаксируем и только затем сохраняем рабочую траекторию.

Баростат намеренно отсутствует: это NVT с постоянной кубической ячейкой. Приведённый ниже unwrapping рассчитан именно на такую геометрию; его нельзя без изменений переносить на NPT или произвольный триклинный бокс.

In [ ]:
%pip -q install openmm==8.3.1 numpy matplotlib scipy

In [ ]:
import time, itertools
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle, Arc
from IPython.display import display, HTML
import openmm as mm
from openmm import app, unit

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'animation.embed_limit': 40})
SEED = 2026
L = 1.8                         # нм, длина ребра
T = 300.0                       # K
DT = 0.002                      # пс, 2 фс
SAVE_EVERY = 10                  # кадр каждые 0.02 пс
EQUIL_PS, PROD_PS = 20.0, 60.0
R = unit.MOLAR_GAS_CONSTANT_R.value_in_unit(
    unit.kilojoule_per_mole/unit.kelvin)
print('OpenMM:', mm.__version__)
print('Платформы:', [mm.Platform.getPlatform(i).getName()
                      for i in range(mm.Platform.getNumPlatforms())])

### 2. Топология и параметры TIP3P

У каждой молекулы три атома и две химические связи. Геометрия: $r_{OH}=0.09572$ нм, $\theta_{HOH}=104.52^\circ$. При `rigidWater=True` OpenMM фиксирует два расстояния O–H **и расстояние H–H**, что фиксирует и угол. Линия H–H на схеме обозначает ограничение, а не третью химическую связь.

| Атом | Масса, Да | Заряд, $e$ | $\sigma$, нм | $\epsilon$, кДж/моль |
|---|---:|---:|---:|---:|
| O | 15.99943 | −0.834 | 0.31507524 | 0.635968 |
| H1, H2 | 1.007947 | +0.417 | 1.0* | 0 |

\* В XML для H записано $\sigma=1$ нм, но при $\epsilon=0$ LJ-взаимодействие с H равно нулю независимо от этого значения. Не следует трактовать его как физический размер водорода.

Для разных молекул взаимодействия описываются кулоновским и LJ-членами:

$$
U_{ij}=\frac{1}{4\pi\epsilon_0}\frac{q_iq_j}{r_{ij}}
+4\epsilon_{ij}\left[\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12}
-\left(\frac{\sigma_{ij}}{r_{ij}}\right)^6\right],
$$

$$
\sigma_{ij}=\frac{\sigma_i+\sigma_j}{2},\qquad
\epsilon_{ij}=\sqrt{\epsilon_i\epsilon_j}.
$$

Для периодической электростатики эта запись понимается как сумма по образам, вычисляемая PME, а не только как взаимодействие с ближайшим образом. Для LJ используем cutoff и стандартную дисперсионную поправку OpenMM. Внутримолекулярные невалентные взаимодействия исключены.

В `tip3p.xml` также есть гармонические параметры $k_b=462750.4$ кДж/(моль·нм$^2$) и $k_\theta=836.8$ кДж/(моль·рад$^2$), с соглашением $U=\frac12 k(q-q_0)^2$. Здесь геометрия жёсткая: колебания связей и угла не интегрируются. Не следует одновременно интерпретировать воду как гибкую молекулу и как жёсткое тело.

In [ ]:
ff = app.ForceField('tip3p.xml')
modeller = app.Modeller(app.Topology(), [] * unit.nanometer)
modeller.addSolvent(ff, model='tip3p', boxSize=mm.Vec3(L, L, L)*unit.nanometer)
topology = modeller.topology
atoms = list(topology.atoms())
waters = [[a.index for a in res.atoms()] for res in topology.residues()]
# Явно используем химические имена, не полагаемся на порядок атомов.
mol = np.array([[next(a.index for a in res.atoms() if a.name == name)
                 for name in ('O', 'H1', 'H2')] for res in topology.residues()])
O = mol[:, 0]
H = mol[:, 1:].ravel()
N = len(O)
system = ff.createSystem(topology, nonbondedMethod=app.PME,
                         nonbondedCutoff=0.8*unit.nanometer,
                         constraints=app.HBonds, rigidWater=True,
                         ewaldErrorTolerance=1e-4, removeCMMotion=True)
nonbonded = next(f for f in system.getForces() if isinstance(f, mm.NonbondedForce))
nonbonded.setUseDispersionCorrection(True)
mass_da = sum(system.getParticleMass(i).value_in_unit(unit.dalton)
              for i in range(system.getNumParticles()))
rho = mass_da * 1.66053906660e-3 / L**3  # г/см^3
print(f'{N} молекул, {len(atoms)} атомов, {topology.getNumBonds()} связей')
print(f'Бокс: {L:.2f} нм; плотность: {rho:.3f} г/см³')
print('Ограничения:', system.getNumConstraints(), '(ожидается 3 на молекулу)')
print('Силовые объекты:', [type(f).__name__ for f in system.getForces()])
print('Атом   масса/Да      q/e       sigma/нм   epsilon/(кДж/моль)')
for i in mol[0]:
    q, sig, eps = nonbonded.getParticleParameters(int(i))
    print(f'{atoms[i].name:4s}  {system.getParticleMass(int(i)).value_in_unit(unit.dalton):9.6f}'
          f'  {q.value_in_unit(unit.elementary_charge):8.3f}'
          f'  {sig.value_in_unit(unit.nanometer):10.7f}'
          f'  {eps.value_in_unit(unit.kilojoule_per_mole):10.6f}')
assert system.getNumConstraints() == 3*N
assert abs(sum(nonbonded.getParticleParameters(i)[0].value_in_unit(unit.elementary_charge)
               for i in range(len(atoms)))) < 1e-8
print('Ограничения первой воды (индексы атомов, расстояние):')
for i in range(system.getNumConstraints()):
    a, b, distance = system.getConstraintParameters(i)
    if a in mol[0] and b in mol[0]: print(a, b, distance)

In [ ]:
rOH = 0.09572
angle = np.deg2rad(104.52)
xy = np.array([[0, 0], [rOH*np.cos(angle/2), rOH*np.sin(angle/2)],
               [rOH*np.cos(angle/2), -rOH*np.sin(angle/2)]])
fig, ax = plt.subplots(figsize=(8, 4))
for j in (1, 2):
    ax.plot(xy[[0,j],0], xy[[0,j],1], c='#475569', lw=4)
ax.plot(xy[1:,0], xy[1:,1], '--', c='#94a3b8', label='H–H: constraint, not a bond')
ax.scatter(*xy[0], s=1300, c='#dc2626', zorder=3)
ax.scatter(xy[1:,0], xy[1:,1], s=550, c='#dbeafe', edgecolor='#2563eb', zorder=3)
for j, label in enumerate(['O', 'H1', 'H2']):
    ax.text(*xy[j], label, ha='center', va='center', zorder=4)
ax.text(-.065, 0, '−0.834 e', ha='center')
ax.text(.10, .075, '+0.417 e'); ax.text(.10, -.08, '+0.417 e')
ax.text(.025, .047, '0.09572 nm', rotation=50, ha='center')
ax.add_patch(Arc((0,0), .07, .07, theta1=-52.26, theta2=52.26, color='#0f766e'))
ax.text(.043, 0, '104.52°', va='center', color='#0f766e')
ax.set(xlim=(-.12,.19), ylim=(-.12,.12), aspect='equal', title='TIP3P: topology and rigid geometry')
ax.axis('off'); ax.legend(loc='lower left'); plt.show()

initial = np.asarray(modeller.positions.value_in_unit(unit.nanometer))
# Для рисунка выбираем ячейку [0,L): переносим каждую воду целиком по O.
initial_view = initial.copy()
for indices in mol:
    initial_view[indices] -= L*np.floor(initial_view[indices[0]]/L)
fig = plt.figure(figsize=(7,6)); ax = fig.add_subplot(projection='3d')
for indices in mol:
    p = initial_view[indices]
    for j in (1,2): ax.plot(*p[[0,j]].T, c='#94a3b8', lw=.5)
ax.scatter(*initial_view[O].T, c='#dc2626', s=13, label='O')
ax.scatter(*initial_view[H].T, c='#60a5fa', s=4, label='H')
for a in itertools.product([0,L], repeat=3):
    for d in range(3):
        b = list(a); b[d] = L-b[d]
        ax.plot(*np.array([a,b]).T, c='black', lw=.5, alpha=.3)
ax.set(xlabel='x / nm', ylabel='y / nm', zlabel='z / nm',
       title=f'Initial water box (whole molecules): {N} waters')
ax.set_box_aspect((1,1,1)); ax.legend(); plt.show()

### 3. Минимизация, релаксация и рабочая траектория

Малый бокс нужен для быстрого занятия. Его плотность задаётся числом молекул и объёмом, а не подстраивается под давление. Cutoff меньше $L/2$. PME учитывает дальнодействующую электростатику; рисунки соседних ячеек ниже — способ визуализации, а не ручное добавление новых частиц в System.

Сохраняем **исходные координаты `raw`** с `enforcePeriodicBox=False`: это координаты, возвращённые Context без дополнительного укладывания в ячейку. Не путайте их с начальным кадром `initial`. На другой платформе или после обработки файла представление исходных координат может отличаться, поэтому непрерывность проверяется отдельно.

`enforcePeriodicBox=True` укладывает **целые молекулы по их центрам**, а не каждый атом независимо. Наш `wrapped = raw % L` ниже — намеренно другое, поатомное представление, удобное для демонстрации скачков.

На начальном 3D-рисунке молекулы перенесены целиком в выбранную ячейку $[0,L)$ по координатам O. Исходная заготовка Modeller может использовать другой центр ячейки; выбор начала координат не меняет периодическую систему. Водороды собранных молекул могут немного выходить за грани.

In [ ]:
available = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
platform_name = next(name for name in ('CUDA', 'OpenCL', 'CPU', 'Reference') if name in available)
platform = mm.Platform.getPlatformByName(platform_name)
properties = {'Precision': 'mixed'} if platform_name in ('CUDA', 'OpenCL') else (
    {'Threads': '2'} if platform_name == 'CPU' else {})
integrator = mm.LangevinMiddleIntegrator(T*unit.kelvin, 1/unit.picosecond, DT*unit.picosecond)
integrator.setRandomNumberSeed(SEED)
integrator.setConstraintTolerance(1e-6)
sim = app.Simulation(topology, system, integrator, platform, properties)
sim.context.setPositions(modeller.positions)
sim.minimizeEnergy(maxIterations=500)
sim.context.setVelocitiesToTemperature(T*unit.kelvin, SEED)
print('Платформа:', platform_name, '; релаксация...')
tstart = time.perf_counter()
sim.step(round(EQUIL_PS/DT))
nsteps = round(PROD_PS/DT)
nframes = nsteps//SAVE_EVERY + 1
raw = np.empty((nframes, len(atoms), 3))
temperature = np.empty(nframes)
energy = np.empty(nframes)
dof = 3*len(atoms) - system.getNumConstraints() - 3  # CMMotionRemover
for frame in range(nframes):
    if frame: sim.step(SAVE_EVERY)
    st = sim.context.getState(getPositions=True, getEnergy=True, enforcePeriodicBox=False)
    raw[frame] = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    energy[frame] = st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)/N
    temperature[frame] = 2*st.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)/(dof*R)
    if frame % 1000 == 0: print(f'Сохранено {frame}/{nframes-1} кадров')
times = np.arange(nframes)*SAVE_EVERY*DT
print(f'Готово за {time.perf_counter()-tstart:.1f} с; средняя T = {temperature.mean():.1f} K')
assert np.isfinite(raw).all()
fig, axes = plt.subplots(1,2,figsize=(10,3))
axes[0].plot(times, temperature, lw=.6); axes[0].axhline(T,c='k',ls='--')
axes[0].set(xlabel='Time / ps', ylabel='T / K', title='Production temperature')
axes[1].plot(times, energy, lw=.6)
axes[1].set(xlabel='Time / ps', ylabel='U / (kJ mol⁻¹ per water)', title='Potential energy')
plt.tight_layout(); plt.show()

### 4. Wrapped, unwrapped и минимальный образ

Для кубической ячейки по каждой декартовой компоненте:

$$
x^{\mathrm{w}}=x-L\left\lfloor\frac{x}{L}\right\rfloor,\qquad 0\le x^{\mathrm{w}}<L.
$$

Если частица пересекает правую грань, $x^{\mathrm{w}}$ меняется с $L-\varepsilon$ на $\varepsilon$. Значение координаты скачет примерно на $-L$, хотя частица прошла только $2\varepsilon$.

Восстановим непрерывную траекторию по последовательным кадрам:

$$
\delta\mathbf r_t=\mathbf r^{\mathrm w}_{t+1}-\mathbf r^{\mathrm w}_{t},\qquad
\delta\mathbf r_t^{\mathrm{MIC}}=\delta\mathbf r_t-L\,\mathrm{round}(\delta\mathbf r_t/L),
$$

$$
\mathbf r^{\mathrm u}_{t+1}=\mathbf r^{\mathrm u}_{t}+\delta\mathbf r_t^{\mathrm{MIC}}.
$$

**Условие:** настоящее перемещение по каждой компоненте между сохранёнными кадрами должно быть меньше $L/2$. При слишком редкой записи неизвестно, сколько раз частица пересекла границу. Малость восстановленного приращения сама по себе этого не доказывает; здесь мы дополнительно сравним результат с исходной непрерывной траекторией.

Начальный периодический образ выбирается произвольно: unwrapped определены с точностью до постоянного вектора $L\mathbf n$ для каждой частицы. Для MSD этот постоянный сдвиг несущественен.

In [ ]:
def mic(delta):
    return delta - L*np.rint(delta/L)

wrapped = np.mod(raw, L)
def unwrap_fixed_box(w):
    u = np.empty_like(w)
    u[0] = w[0]
    u[1:] = u[0] + np.cumsum(mic(np.diff(w, axis=0)), axis=0)
    return u

unwrapped = unwrap_fixed_box(wrapped)
# Совмещаем только начальные образы для сравнения с Context.
u_aligned = unwrapped + (raw[0]-unwrapped[0])[None,:,:]
raw_step = np.max(np.abs(np.diff(raw[:,O], axis=0)))
unwrap_error = np.max(np.abs(u_aligned[:,O]-raw[:,O]))
print(f'Max component of raw O step = {raw_step:.6g} nm; L/2 = {L/2:.3f} nm')
print(f'Max unwrapped–raw O difference = {unwrap_error:.3g} nm')
assert raw_step < L/2, 'Исходные координаты переупакованы или запись слишком редкая.'
assert unwrap_error < 2e-4, 'Проверьте соглашение о координатах данной платформы.'

# Для рисунков переносим целую молекулу по её O; связи не рвутся.
# H могут немного выходить за грань — это ожидаемо.
whole = np.empty_like(raw)
for oxygen, h1, h2 in mol:
    whole[:,oxygen] = wrapped[:,oxygen]
    for h in (h1,h2):
        whole[:,h] = whole[:,oxygen] + mic(raw[:,h]-raw[:,oxygen])

# Выбираем реальное пересечение x- или y-грани внутри траектории.
crossings = np.argwhere(np.abs(np.diff(wrapped[:,O,:2],axis=0)) > L/2)
assert len(crossings), 'Увеличьте PROD_PS: пока нет пересечений.'
interior = crossings[(crossings[:,0] > 30) & (crossings[:,0] < nframes-31)]
event, water_index, component = (interior[len(interior)//2] if len(interior) else crossings[0])
tracked = O[water_index]
print(f'Выбран O #{tracked}, crossing at t = {times[event+1]:.3f} ps')
fig, axes = plt.subplots(1,2,figsize=(11,3.5))
sl = slice(max(0,event-35), min(nframes,event+36))
for arr, label, style in [(raw,'raw','-'),(wrapped,'wrapped','--'),(u_aligned,'unwrapped',':')]:
    axes[0].plot(times[sl],arr[sl,tracked,component],style,label=label,lw=2)
axes[0].set(xlabel='Time / ps',ylabel=f'{"xy"[component]} / nm',title='One boundary crossing')
axes[0].legend()
axes[1].plot(times[1:][sl], np.linalg.norm(np.diff(wrapped[:,tracked],axis=0),axis=1)[sl],label='wrapped difference')
axes[1].plot(times[1:][sl], np.linalg.norm(np.diff(unwrapped[:,tracked],axis=0),axis=1)[sl],label='unwrapped difference')
axes[1].set(xlabel='Time / ps',ylabel='Step length / nm',title='Coordinate jump ≠ physical displacement')
axes[1].legend(); plt.tight_layout(); plt.show()

### 5. Анимация: одна частица, четыре представления

Красная молекула одна и та же во всех панелях. Синие точки — кислороды остальных молекул. Показана проекция **XY**, поэтому точки могут накладываться, хотя различаются по $z$. Панели raw и unwrapped могут совпадать — в этом запуске Context сохраняет непрерывные координаты, а не создаёт третью физическую траекторию.

Во второй панели каждый атом завёрнут независимо: около грани даже геометрия воды визуально «разрывается». В четвёртой панели молекула собрана целиком, показаны 8 соседних XY-образов. Это срез представления периодической решётки; полное 3D-окружение содержит также образы по $z$. Образы не являются независимыми молекулами и не должны включаться в подсчёт числа частиц или MSD.

Анимация использует короткий фрагмент около **реального** пересечения границы, без добавленного дрейфа или искусственной скорости.

In [ ]:
start, stop = max(0,event-45), min(nframes,event+46)
frames = np.arange(start, stop, 2)
ids = mol[water_index]
fig, axs = plt.subplots(2,2,figsize=(10,9), dpi=75)
axs = axs.ravel()
shifts = np.array(list(itertools.product([-1,0,1],repeat=2))) * L

def animate(f):
    for ax in axs: ax.clear()
    arrays = [raw, wrapped, u_aligned, whole]
    titles = ['Raw: Context coordinates', 'Wrapped: every atom in [0,L)',
              'Unwrapped: reconstructed path', 'Whole molecule + periodic images']
    for panel, (ax, arr, title) in enumerate(zip(axs, arrays, titles)):
        p = arr[f]
        if panel == 3:
            for shift in shifts:
                central = np.all(shift == 0)
                alpha = .55 if central else .13
                ax.scatter(p[O,0]+shift[0], p[O,1]+shift[1], s=6,c='#2563eb',alpha=alpha)
                water = p[ids,:2] + shift
                for j in (1,2): ax.plot(water[[0,j],0],water[[0,j],1],c='#dc2626',lw=1.7,alpha=.9)
                ax.scatter(*water[0],s=55,c='#dc2626',zorder=5)
                ax.scatter(water[1:,0],water[1:,1],s=18,c='#fbbf24',zorder=5)
                ax.add_patch(Rectangle(shift,L,L,fill=False,lw=.6,color='gray'))
            ax.set(xlim=(-.45*L,1.45*L),ylim=(-.45*L,1.45*L))
        else:
            ax.scatter(p[O,0],p[O,1],s=7,c='#2563eb',alpha=.35)
            water = p[ids,:2]
            # Wrapped bonds intentionally omitted: drawing long lines is misleading.
            if panel != 1:
                for j in (1,2): ax.plot(water[[0,j],0],water[[0,j],1],c='#dc2626',lw=2)
            ax.scatter(*water[0],s=65,c='#dc2626',zorder=5)
            ax.scatter(water[1:,0],water[1:,1],s=22,c='#fbbf24',zorder=5)
            trace = arr[max(start,f-25):f+1,tracked,:2].copy()
            if panel == 1 and len(trace)>1:
                jumps = np.any(np.abs(np.diff(trace,axis=0))>L/2,axis=1)
                trace[1:][jumps] = np.nan  # не рисуем ложный путь сквозь ячейку
            ax.plot(trace[:,0],trace[:,1],c='#dc2626',lw=1.3)
            if panel == 1: ax.set(xlim=(-.1*L,1.1*L),ylim=(-.1*L,1.1*L))
            else:
                trajectory = arr[frames,tracked,:2]
                low=np.minimum(trajectory.min(axis=0)-.15,0)
                high=np.maximum(trajectory.max(axis=0)+.15,L)
                ax.set(xlim=(low[0]-.05,high[0]+.05),ylim=(low[1]-.05,high[1]+.05))
        ax.add_patch(Rectangle((0,0),L,L,fill=False,lw=2,color='black'))
        ax.set(title=title,xlabel='x / nm',ylabel='y / nm',aspect='equal')
        ax.text(.02,.98,f'O: ({p[tracked,0]:.3f}, {p[tracked,1]:.3f}, {p[tracked,2]:.3f}) nm',
                transform=ax.transAxes,va='top',fontsize=8,
                bbox={'facecolor':'white','alpha':.85,'edgecolor':'none'})
    fig.suptitle(f'Same water molecule · t = {times[f]:.3f} ps',fontsize=14)
    fig.tight_layout(rect=(0,0,1,.96))

ani = FuncAnimation(fig, animate, frames=frames, interval=100, repeat=True)
html_animation = ani.to_jshtml()
animate(int(event)+1)
plt.show()  # статичный кадр доступен даже без выполнения JavaScript
plt.close(fig)
display(HTML(html_animation))

### 6. Прямая проверка: энергия и силы не скачут

Перенесём одну **целую** молекулу на $L$ вдоль $x$ и пересчитаем энергию и силы, не делая ни одного шага динамики. При PBC это другая запись того же состояния. Разности должны быть малы и определяться численной точностью PME и платформы. Это более строгая проверка, чем сравнение соседних кадров MD: между кадрами система действительно движется и энергия может меняться.

In [ ]:
saved = sim.context.getState(getPositions=True).getPositions(asNumpy=True)
p = saved.value_in_unit(unit.nanometer).copy()
s0 = sim.context.getState(getEnergy=True, getForces=True)
p[ids,0] += L
try:
    sim.context.setPositions(p*unit.nanometer)
    s1 = sim.context.getState(getEnergy=True, getForces=True)
    du = (s1.getPotentialEnergy()-s0.getPotentialEnergy()).value_in_unit(unit.kilojoule_per_mole)
    df = (s1.getForces(asNumpy=True)-s0.getForces(asNumpy=True)).value_in_unit(
        unit.kilojoule_per_mole/unit.nanometer)
finally:
    sim.context.setPositions(saved)
print(f'ΔU (all waters) = {du:.6g} kJ/mol')
print(f'RMS ΔF = {np.sqrt(np.mean(df**2)):.6g} kJ/(mol nm)')

### 7. MSD двумя способами: правильный и ошибочный

Среднеквадратичное смещение кислородов с усреднением по молекулам и временам начала:

$$
\mathrm{MSD}(\tau)=\frac{1}{N(N_t-m)}\sum_{i=1}^{N}\sum_{t=0}^{N_t-m-1}
\left|\mathbf r_i(t+m)-\mathbf r_i(t)\right|^2,\quad \tau=m\Delta t.
$$

1. **Правильно:** используем unwrapped, суммирующие все пересечения границ.
2. **Неправильно:** прямо вычитаем wrapped-координаты. Частицы около противоположных граней дают искусственно большие перемещения даже при малом $\tau$; на больших временах такой MSD ограничен размером ячейки.

Для двух независимых равномерно распределённых точек куба ошибочный wrapped-MSD стремится к $L^2/2$. Это ориентир предела полной декорреляции, а не обязательное плато короткой траектории.

Применить MIC один раз к разности **конечных** координат также недостаточно для диффузии: так теряется число обходов ячейки. MIC применяется к **последовательным коротким приращениям**, которые затем накапливаются.

В 3D диффузионном режиме $\mathrm{MSD}(\tau)\approx6D\tau+C$. Ниже рассчитывается только грубая оценка наклона. Короткая траектория, небольшой периодический бокс, модель TIP3P, термостат и выбор окна влияют на результат.

In [ ]:
lags = np.unique(np.r_[0, np.geomspace(1, (nframes-1)//3, 100).astype(int)])
def time_origin_msd(x, lags):
    return np.array([0.0 if lag==0 else np.mean(np.sum((x[lag:]-x[:-lag])**2,axis=-1))
                     for lag in lags])
msd_u = time_origin_msd(unwrapped[:,O],lags)
msd_w = time_origin_msd(wrapped[:,O],lags)

tau = lags*SAVE_EVERY*DT
fit = (tau >= 5) & (tau <= 15)
slope, intercept = np.polyfit(tau[fit],msd_u[fit],1)
D = slope/6
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(tau,msd_u,label='Unwrapped: correct',lw=2)
ax.plot(tau,msd_w,label='Wrapped: incorrect',lw=2)
ax.set(xlabel='Lag time / ps',ylabel='MSD / nm²'); ax.legend()
ax.axhline(L**2/2,ls=':',color='gray',label='L²/2')
ax.plot(tau[fit],slope*tau[fit]+intercept,'k--',label='Fit 5–15 ps')
ax.legend(); ax.set_title('All sampled lag times')
plt.tight_layout(); plt.show()
print(f'Учебная оценка D: {D:.5f} nm²/ps = {D*1e-6:.3g} m²/s')
print('Окно фитирования: 5–15 пс.')

### 7A. Два подхода к коэффициенту самодиффузии: Эйнштейн и Грин–Кубо

Сначала используем готовую **NVT-траекторию**, затем запускаем отдельную **NVE-траекторию** из её конечного равновесного состояния. Модель, объём и число молекул одинаковы.

| Подход | Данные | Учебный протокол | Что выбираем при анализе |
|---|---|---|---|
| Эйнштейн | Unwrapped-координаты центров масс | NVT, LangevinMiddle, 300 K | Линейный участок MSD |
| Грин–Кубо | Скорости центров масс, частая запись | NVE без термостата и баростата | Область плато накопленного интеграла VACF |

**Различие ансамблей не является ограничением формул.** Обе формулы связаны кинематически для одной и той же стационарной динамики. Эйнштейна можно применять в NVE, а VACF можно измерять в NVT. Однако термостат Ланжевена изменяет динамику и, следовательно, измеряемый $D$. Для оценки транспорта при невозмущённой ньютоновской динамике после термализации обычно отключают термостат. Поэтому совпадение NVT- и NVE-оценок здесь не обязательно должно выполняться точно. Даже при достаточной статистике они могут различаться из-за термостата, температуры и конечного размера.

В предыдущем разделе MSD считался по O. Для прямого сравнения двух подходов используем **одну и ту же величину — центр масс целой молекулы воды**:

$$
\mathbf R_i=\frac{\sum_{a\in i}m_a\mathbf r_a}{M_w},\qquad
\mathbf V_i=\frac{\sum_{a\in i}m_a\mathbf v_a}{M_w}.
$$

Так мы отделяем поступательное движение молекулы от вращательного движения атомов вокруг её центра масс. На больших временах O и центр масс имеют одинаковый коэффициент поступательной диффузии; коротковременные MSD и VACF различаются. Общий перенос всего бокса вычитаем, анализируя движение относительно центра масс системы.

Для изотропной системы в трёх измерениях:

$$
C_{VV}(\tau)=\frac1N\sum_i\langle\mathbf V_i(t+\tau)\cdot\mathbf V_i(t)\rangle_t,
\qquad
D_{\mathrm{GK}}=\frac13\int_0^\infty C_{VV}(\tau)\,d\tau.
$$

В $C_{VV}$ уже суммируются три декартовы компоненты; множитель $1/3$ применяется **только один раз**. При интегрировании нормированной VACF необходимо вернуть амплитуду:

$$
D=\frac{C_{VV}(0)}3\int_0^\infty\frac{C_{VV}(\tau)}{C_{VV}(0)}\,d\tau.
$$

Формула Эйнштейна:

$$
D_E=\frac16\lim_{\tau\to\infty}\frac{d}{d\tau}\mathrm{MSD}(\tau),\qquad
\mathrm{MSD}(\tau)\simeq6D_E\tau+B.
$$

На конечном интервале подбираем наклон **со свободным интерсептом** $B$, не заставляя диффузионную прямую проходить через начало координат. Коротковременный баллистический режим не входит в фитирование.

#### Почему формулы эквивалентны?

Из $\Delta\mathbf R(t)=\int_0^t\mathbf V(s)\,ds$, стационарности и симметрии корреляционной функции следует

$$
\mathrm{MSD}(t)=2\int_0^t(t-\tau)C_{VV}(\tau)\,d\tau,
$$

$$
\frac16\frac{d\,\mathrm{MSD}(t)}{dt}
=\frac13\int_0^t C_{VV}(\tau)\,d\tau
\equiv D_{\mathrm{GK}}(t).
$$

При затухании корреляций правая часть выходит на плато, а MSD становится линейным.

In [ ]:
# Центры масс: сначала собираем H вокруг unwrapped O, затем усредняем.
masses = np.array([system.getParticleMass(i).value_in_unit(unit.dalton)
                   for i in range(system.getNumParticles())])
water_masses = masses[mol]  # N × 3
Mwater = water_masses.sum(axis=1)
weights = water_masses/Mwater[:,None]

def water_com(positions):
    # positions: (..., n_atoms, 3). Кислород может быть в любом образе.
    oxygen = positions[...,O,:]
    offsets = mic(positions[...,mol,:] - oxygen[..., :,None,:])
    return oxygen + np.sum(offsets*weights[...,None],axis=-2)

com_nvt = water_com(raw)
# Исправление на перенос системы: координаты в системе её центра масс.
com_nvt -= np.average(com_nvt,axis=1,weights=Mwater)[:,None,:]
msd_com_nvt = time_origin_msd(com_nvt,lags)
fit_nvt = (tau>=5)&(tau<=15)
slope_nvt, intercept_nvt = np.polyfit(tau[fit_nvt],msd_com_nvt[fit_nvt],1)
D_einstein_nvt = slope_nvt/6
print(f'Einstein / NVT / COM: {D_einstein_nvt:.5f} nm²/ps = {D_einstein_nvt*1e-6:.3g} m²/s')
print('Чувствительность NVT-оценки к окну:')
for lo,hi in [(3,10),(5,15),(8,18)]:
    mask = (tau>=lo)&(tau<=hi)
    d = np.polyfit(tau[mask],msd_com_nvt[mask],1)[0]/6
    print(f'  {lo:2d}–{hi:2d} ps: D = {d:.5f} nm²/ps')
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(tau,msd_com_nvt,label='NVT: whole-water COM')
ax.plot(tau[fit_nvt],slope_nvt*tau[fit_nvt]+intercept_nvt,'k--',label='Fit 5–15 ps')
ax.axvspan(5,15,alpha=.1,color='green')
ax.set(xlabel='Lag time / ps',ylabel='MSD / nm²',title='Einstein estimate in NVT')
ax.legend(); plt.show()

### 7B. NVE после термализации: сохранение энергии и частая запись скоростей

Копируем System, удаляем `CMMotionRemover` и создаём **новый Context** без термостата. Начальные координаты и скорости берём из конца NVT, один раз убираем общий импульс и накладываем ограничения. Во время NVE скорости не пересэмплируются и не масштабируются.

Используем **velocity Verlet с ограничениями RATTLE** через `CustomIntegrator`. Координаты и скорости относятся к одному моменту, что удобно для VACF и проверки энергии. Обычный `VerletIntegrator` OpenMM использует leapfrog, где они сдвинуты на полшага; здесь это соглашение специально обходим явной реализацией velocity Verlet.

Шаг NVE — 1 фс, запись каждые 2 фс: VACF воды меняется на десятках фемтосекунд, поэтому интервал 0.02 пс из MSD-раздела для её формы уже слишком груб. После переключения отбрасываем 2 пс, затем записываем 40 пс. Это добавляет время вычислений и примерно несколько десятков МБ массивов. Температура NVE **флуктуирует**, а её среднее не обязано быть ровно 300 K.

Диагностика: график $(E(t)-E(0))/N$, диапазон колебаний и линейный дрейф. Если дрейф заметен на фоне флуктуаций, уменьшите шаг, ужесточите PME/constraints и удлините проверку. NVE означает численное приближение к постоянной энергии, а не её побитовое постоянство.

In [ ]:
NVE_DT = 0.001       # пс
NVE_SAVE = 2         # шагов; 0.002 пс между скоростями
NVE_SETTLE = 2.0     # пс
NVE_PROD = 40.0      # пс
system_nve = mm.XmlSerializer.deserialize(mm.XmlSerializer.serialize(system))
for i in reversed(range(system_nve.getNumForces())):
    if isinstance(system_nve.getForce(i), mm.CMMotionRemover):
        system_nve.removeForce(i)
assert not any(isinstance(f, mm.CMMotionRemover) for f in system_nve.getForces())

# RATTLE velocity Verlet; x_trial нужен для поправки импульса от constraints.
nve_integrator = mm.CustomIntegrator(NVE_DT*unit.picosecond)
nve_integrator.addPerDofVariable('x_trial',0)
nve_integrator.addComputePerDof('v','v+0.5*dt*f/m')
nve_integrator.addComputePerDof('x','x+dt*v')
nve_integrator.addComputePerDof('x_trial','x')
nve_integrator.addConstrainPositions()
nve_integrator.addComputePerDof('v','v+(x-x_trial)/dt+0.5*dt*f/m')
nve_integrator.addConstrainVelocities()
nve_integrator.setConstraintTolerance(1e-7)
# Default kinetic energy: sum(m*v*v/2), velocities at full time steps.
nve_integrator.setKineticEnergyExpression('m*v*v/2')
sim_nve = app.Simulation(topology,system_nve,nve_integrator,platform,properties)
seed_state = sim.context.getState(getPositions=True,getVelocities=True)
sim_nve.context.setPeriodicBoxVectors(*seed_state.getPeriodicBoxVectors())
sim_nve.context.setPositions(seed_state.getPositions())
v0 = seed_state.getVelocities(asNumpy=True).value_in_unit(unit.nanometer/unit.picosecond)
v0 -= np.average(v0,axis=0,weights=masses)
sim_nve.context.setVelocities(v0*unit.nanometer/unit.picosecond)
sim_nve.context.applyConstraints(1e-7)
sim_nve.context.applyVelocityConstraints(1e-7)
t0_nve = time.perf_counter()
sim_nve.step(round(NVE_SETTLE/NVE_DT))
n_nve = round(NVE_PROD/(NVE_DT*NVE_SAVE))+1
vcom_nve = np.empty((n_nve,N,3),dtype=np.float32)
rcom_nve = np.empty((n_nve,N,3),dtype=np.float64)
bulk_v_nve = np.empty((n_nve,3))
E_nve = np.empty(n_nve); T_nve = np.empty(n_nve)
for f in range(n_nve):
    if f: sim_nve.step(NVE_SAVE)
    st = sim_nve.context.getState(getPositions=True,getVelocities=True,getEnergy=True,
                                  enforcePeriodicBox=False)
    pos = st.getPositions(asNumpy=True).value_in_unit(unit.nanometer)
    vel = st.getVelocities(asNumpy=True).value_in_unit(unit.nanometer/unit.picosecond)
    rcom_nve[f] = water_com(pos)
    vc = np.sum(vel[mol]*weights[:,:,None],axis=1)
    bulk_v_nve[f] = np.average(vc,axis=0,weights=Mwater)
    vcom_nve[f] = vc-bulk_v_nve[f]  # система отсчёта общего центра масс
    kinetic = st.getKineticEnergy().value_in_unit(unit.kilojoule_per_mole)
    E_nve[f] = kinetic + st.getPotentialEnergy().value_in_unit(unit.kilojoule_per_mole)
    T_nve[f] = 2*kinetic/(dof*R)
    if f%5000==0: print(f'NVE: {f}/{n_nve-1} frames')
dt_v = NVE_DT*NVE_SAVE
t_nve = np.arange(n_nve)*dt_v
assert np.max(np.abs(np.diff(rcom_nve,axis=0))) < L/2
rcom_nve -= np.average(rcom_nve,axis=1,weights=Mwater)[:,None,:]
assert np.isfinite(vcom_nve).all() and np.isfinite(E_nve).all()
drift = np.polyfit(t_nve,E_nve/N,1)[0]
print(f'NVE elapsed: {time.perf_counter()-t0_nve:.1f} s')
print(f'Mean NVE T: {T_nve.mean():.2f} K; NVT mean T: {temperature.mean():.2f} K')
print(f'Energy drift: {drift:.3g} kJ/(mol water ps); range: {np.ptp(E_nve)/N:.3g} kJ/mol water')
print(f'Max bulk speed: {np.linalg.norm(bulk_v_nve,axis=1).max():.3g} nm/ps')
fig, axes = plt.subplots(1,2,figsize=(11,3.5))
axes[0].plot(t_nve,(E_nve-E_nve[0])/N,lw=.5)
axes[0].set(xlabel='Time / ps',ylabel='ΔE / (kJ mol⁻¹ per water)',title='NVE: energy conservation')
axes[1].plot(t_nve,T_nve,lw=.5)
axes[1].axhline(T_nve.mean(),c='k',ls='--',label=f'Mean {T_nve.mean():.1f} K')
axes[1].set(xlabel='Time / ps',ylabel='T / K',title='NVE: temperature fluctuates')
axes[1].legend(); plt.tight_layout(); plt.show()

### 7C. VACF и накопленный интеграл Грина–Кубо

Оценка по множеству временных начал:

$$
\widehat C_{VV}(m\Delta t)=\frac{1}{N(M-m)}
\sum_{i=1}^N\sum_{t=0}^{M-m-1}\mathbf V_i(t+m)\cdot\mathbf V_i(t).
$$

Вычисляем её через FFT с дополнением нулями, чтобы получить **линейную**, а не циклическую корреляцию. Делитель $M-m$ учитывает число доступных временных начал для каждого лага. Проверяем несколько лагов прямым вычислением. Мы не вычитаем среднюю за траекторию скорость каждой отдельной молекулы: такое вычитание на коротком отрезке искусственно искажает медленный хвост. Удалён только общий перенос системы.

$$
D_{\mathrm{GK}}(t_{\max})=\frac13\int_0^{t_{\max}}\widehat C_{VV}(t)\,dt.
$$

Интегрируем методом трапеций. Отрицательная часть VACF означает обратное рассеяние молекулы окружением и **должна входить со своим знаком**. Нельзя интегрировать только положительную часть или обрезать интеграл при первом нуле.

Показываем четыре временных блока по 10 пс. Их разброс — диагностика выборки, не строгий доверительный интервал. Интервал 1–3 пс задан как **пробное окно плато**, а не автоматически подтверждённый предел: следует смотреть на кривые и менять окно. Не нужно интегрировать шумный хвост до конца траектории только потому, что формальная формула содержит бесконечность.

In [ ]:
from scipy.fft import rfft, irfft, next_fast_len
from scipy.integrate import cumulative_trapezoid

def vacf_fft(velocities,max_lag,chunk=24):
    v = np.asarray(velocities,dtype=np.float64)
    nt,nmol,_ = v.shape
    assert max_lag < nt
    flat = v.reshape(nt,-1)
    nfft = next_fast_len(2*nt-1)
    sums = np.zeros(max_lag+1)
    for start in range(0,flat.shape[1],chunk):
        vf = rfft(flat[:,start:start+chunk],n=nfft,axis=0)
        ac = irfft(vf.conj()*vf,n=nfft,axis=0)[:max_lag+1]
        sums += ac.sum(axis=1)
    return sums/(nmol*(nt-np.arange(max_lag+1)))

max_lag_v = round(5.0/dt_v)
lag_v = np.arange(max_lag_v+1)*dt_v
vacf = vacf_fft(vcom_nve,max_lag_v)
for lag_check in [0,1,20,100]:
    v = vcom_nve.astype(float)
    direct = np.mean(np.sum(v[lag_check:]*v[:len(v)-lag_check],axis=2))
    assert np.isclose(vacf[lag_check],direct,atol=1e-10,rtol=1e-8)
D_running = cumulative_trapezoid(vacf,dx=dt_v,initial=0)/3
segments = np.array_split(np.arange(n_nve),4)
C_blocks = np.array([vacf_fft(vcom_nve[seg],max_lag_v) for seg in segments])
D_blocks = cumulative_trapezoid(C_blocks,dx=dt_v,axis=1,initial=0)/3
plateau = (lag_v>=1)&(lag_v<=3)
D_gk_nve = D_running[plateau].mean()
D_block_values = D_blocks[:,plateau].mean(axis=1)
print(f'GK / NVE / COM, mean running integral over 1–3 ps: {D_gk_nve:.5f} nm²/ps')
print('Block estimates (nm²/ps):', np.round(D_block_values,5))
print('Block SD (not a confidence interval):',f'{D_block_values.std(ddof=1):.5f} nm²/ps')
print('Чувствительность к верхнему пределу интеграла:')
for tcut in [.2,.5,1,2,3,5]:
    print(f'  {tcut:3.1f} ps: {D_running[round(tcut/dt_v)]:.5f} nm²/ps')
C0_expected = 3*R*T_nve.mean()/Mwater.mean()*(1-1/N)
print(f'C(0) = {vacf[0]:.4f} nm²/ps²; equipartition reference ≈ {C0_expected:.4f}')
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(lag_v,vacf/vacf[0],c='#2563eb')
axes[0].axhline(0,c='gray',ls=':')
axes[0].set(xlim=(0,.5),xlabel='Lag time / ps',ylabel='Cvv(t) / Cvv(0)',title='NVE: molecular COM velocity memory')
for curve in D_blocks: axes[1].plot(lag_v,curve,alpha=.35,lw=1,color='gray')
axes[1].plot(lag_v,D_running,c='#2563eb',lw=2,label='Full NVE trajectory')
axes[1].axvspan(1,3,alpha=.1,color='green',label='Trial plateau window')
axes[1].axhline(D_einstein_nvt,c='#dc2626',ls='--',label='Einstein / NVT')
axes[1].set(xlabel='Integration limit / ps',ylabel='D(t) / (nm² ps⁻¹)',title='Running Green–Kubo integral; gray = blocks')
axes[1].legend(); plt.tight_layout(); plt.show()

### 7D. Сопоставление результатов и проверка связи формул

Для контроля дополнительно считаем MSD по **той же NVE-траектории**, на которой получена VACF. Это помогает отличить расхождение из-за термостата от расхождения двух оценок на одной динамике.

Коротковременный NVE-MSD сравниваем с реконструкцией

$$
\mathrm{MSD}_{\mathrm{VACF}}(t)=2\int_0^t(t-s)C_{VV}(s)\,ds
=6\int_0^t D_{\mathrm{GK}}(s)\,ds.
$$

На конечной выборке кривые могут немного различаться: оценки используют разные наборы временных начал, а стационарность и интегрирование выполнены приближённо. Близость кривых — проверка масштаба, единиц, факторов 3 и 6 и реализации корреляции.

$1$ нм$^2$/пс $=10^{-6}$ м$^2$/с. Не сравнивайте напрямую численные значения в разных единицах.

In [ ]:
# MSD NVE прореживаем уже после записи скоростей: VACF сохраняет шаг 2 фс.
stride_pos = 10
pos_nve_coarse = rcom_nve[::stride_pos]
dt_pos = dt_v*stride_pos
lag_nve = np.unique(np.r_[0,np.geomspace(1,len(pos_nve_coarse)//3,90).astype(int)])
tau_nve = lag_nve*dt_pos
msd_com_nve = time_origin_msd(pos_nve_coarse,lag_nve)
fit_nve = (tau_nve>=5)&(tau_nve<=12)
slope_nve, intercept_nve = np.polyfit(tau_nve[fit_nve],msd_com_nve[fit_nve],1)
D_einstein_nve = slope_nve/6
msd_from_vacf = 6*cumulative_trapezoid(D_running,dx=dt_v,initial=0)
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot(tau,msd_com_nvt,label='NVT / COM')
axes[0].plot(tau_nve,msd_com_nve,label='NVE / COM')
axes[0].plot(tau[fit_nvt],slope_nvt*tau[fit_nvt]+intercept_nvt,'--',c='black')
axes[0].plot(tau_nve[fit_nve],slope_nve*tau_nve[fit_nve]+intercept_nve,':',c='black')
axes[0].set(xlabel='Lag time / ps',ylabel='MSD / nm²',title='Same observable, different dynamics')
axes[0].legend()
axes[1].plot(tau_nve,msd_com_nve,'o',ms=3,label='Direct NVE MSD')
axes[1].plot(lag_v,msd_from_vacf,label='NVE MSD reconstructed from VACF')
axes[1].set(xlim=(0,2),ylim=(0,np.interp(2,lag_v,msd_from_vacf)*1.2),
            xlabel='Lag time / ps',ylabel='MSD / nm²',title='Einstein ↔ Green–Kubo on the same NVE data')
axes[1].legend(); plt.tight_layout(); plt.show()
print('Все оценки относятся к COM воды относительно общего центра масс:')
print(f'Einstein NVT (5–15 ps): {D_einstein_nvt:.5f} nm²/ps = {D_einstein_nvt*1e-6:.3g} m²/s')
print(f'Green–Kubo NVE (trial plateau 1–3 ps): {D_gk_nve:.5f} nm²/ps = {D_gk_nve*1e-6:.3g} m²/s')
print(f'Einstein NVE, control (5–12 ps): {D_einstein_nve:.5f} nm²/ps = {D_einstein_nve*1e-6:.3g} m²/s')
print(f'Mean temperatures: NVT {temperature.mean():.1f} K; NVE {T_nve.mean():.1f} K')
print('Разброс окон/блоков и короткая длина запуска ограничивают точность сравнения.')

#### Как интерпретировать сравнение

- Быстрое затухание VACF показывает потерю памяти начальной скорости; отрицательный участок соответствует возвратному движению в окружении соседей.
- Плато $D_{\mathrm{GK}}(t)$ имеет смысл, если его положение устойчиво к изменению окна и длины траектории. Разброс четырёх серых кривых показывает, насколько это условие выполнено в коротком примере.
- Несовпадение Эйнштейна и Грина–Кубо на **одной NVE-траектории** требует проверки статистики, временного разрешения, хвоста VACF и линейного режима MSD. Подгонять окна только ради совпадения нельзя.
- Несовпадение NVT и NVE дополнительно связано с динамикой термостата и различиями средней температуры. Слабое трение не гарантирует нулевого влияния на транспорт.
- Один старт из NVT даёт одну NVE-энергетическую поверхность. Для более надёжного усреднения нужны несколько независимых стартов. Малый бокс, плотность, периодическая гидродинамика и свойства TIP3P также влияют на $D$; это не точное предсказание экспериментальной диффузии воды.

**Дополнительные задания:** увеличьте время NVE; измените интервал записи скоростей (2, 10, 20 фс); сравните $D_{\mathrm{GK}}$ при разных верхних пределах; повторите NVT с трением 0.1, 1 и 10 пс$^{-1}$; сравните VACF кислорода и COM. В каждом случае меняйте один параметр и проверяйте сходимость.

Источник формул: загруженные слайды `diffusion(1).pdf`, стр. 1 (Грин–Кубо), стр. 2 (Эйнштейн), стр. 3 (связь через интеграл VACF). [Документация интеграторов OpenMM](https://docs.openmm.org/latest/userguide/theory/04_integrators.html) объясняет отличие leapfrog и согласования координат со скоростями.

### 8. RDF: локальная структура воды и периодические расстояния

**Источник данных в этом разделе — исходная NVT-траектория `wrapped`; отдельный NVE-запуск используется в разделах 7B–7D.**

Радиальная функция распределения $g_{AB}(r)$ сравнивает наблюдаемое число соседей B на расстоянии $r$ от A с числом для однородного идеального распределения. Это **пространственная**, а не временная характеристика.

Здесь расстояния вычисляются **в 3D**, независимо от XY-проекции анимации:

$$
\mathbf r_{ij}^{\mathrm{MIC}}=\mathbf r_j-\mathbf r_i
-L\,\mathrm{round}\!\left(\frac{\mathbf r_j-\mathbf r_i}{L}\right),\qquad
r_{ij}=|\mathbf r_{ij}^{\mathrm{MIC}}|.
$$

Для $g_{OO}$ считаем каждую пару один раз ($i<j$). При $M$ кадрах, объёме $V=L^3$ и сферической оболочке

$$
\Delta V_k=\frac{4\pi}{3}(r_{k+1}^3-r_k^3)
$$

используем нормировку

$$
g_{OO}(r_k)=\frac{H_k}{M\,[N(N-1)/2]\,\Delta V_k/V}.
$$

Здесь $H_k$ — общее число попаданий пар в оболочку. Выбранная конечномерная нормировка даёт $g=1$ для независимых частиц даже при конечном $N$. При нормировке через $N^2/2$ идеальный предел был бы $1-1/N$.

Также считаем **межмолекулярные** $g_{OH}$ и $g_{HH}$. Внутримолекулярные пары исключаем, иначе жёсткие расстояния O–H и H–H дадут узкие геометрические пики, не характеризующие соседние молекулы. Число допустимых пар на кадр:

- OO: $N(N-1)/2$;
- OH: $2N(N-1)$, каждый O со всеми H других молекул;
- HH: $2N(N-1)$, неупорядоченные пары H из разных молекул.

**Ограничение:** используем $r_{\max}=L/2$. Только до этого радиуса сферическая оболочка целиком помещается в область минимальных образов. За пределами $L/2$ нельзя использовать ту же сферическую нормировку без геометрической поправки.

Для сравнения намеренно посчитаем ошибочный $g_{OO}$ по прямым расстояниям wrapped-координат. Он теряет близких соседей через границу: особенно заметно подавление распределения на больших $r$.

In [ ]:
edges = np.linspace(0,L/2,151)
r = (edges[:-1]+edges[1:])/2
shell = 4*np.pi/3*np.diff(edges**3)
V = L**3
rdf_frames = np.arange(0,nframes,10)  # каждые 0.2 пс; корреляции всё ещё возможны
io, jo = np.triu_indices(N,k=1)
ih, jh = np.triu_indices(2*N,k=1)
h_molecule = np.repeat(np.arange(N),2)
keep_hh = h_molecule[ih] != h_molecule[jh]
ih, jh = ih[keep_hh], jh[keep_hh]
aoh, boh = np.where(np.arange(N)[:,None] != h_molecule[None,:])
pair_sets = {'OO':(O[io],O[jo]), 'OH':(O[aoh],H[boh]), 'HH':(H[ih],H[jh])}
hist = {name:[] for name in ('OO','OH','HH','OO_no_PBC')}
for f in rdf_frames:
    p = wrapped[f]
    for name,(ia,ib) in pair_sets.items():
        delta = p[ib]-p[ia]
        hist[name].append(np.histogram(np.linalg.norm(mic(delta),axis=1),bins=edges)[0])
        if name=='OO':
            hist['OO_no_PBC'].append(np.histogram(np.linalg.norm(delta,axis=1),bins=edges)[0])
hist = {k:np.asarray(v) for k,v in hist.items()}
npairs = {k:len(v[0]) for k,v in pair_sets.items()}
npairs['OO_no_PBC'] = npairs['OO']
g = {k:v.mean(axis=0)/(npairs[k]*shell/V) for k,v in hist.items()}
# Разброс блочных профилей: НЕ доверительный интервал независимых измерений.
blocks = np.array_split(np.arange(len(rdf_frames)),4)
gblocks = np.array([hist['OO'][b].mean(axis=0)/(npairs['OO']*shell/V) for b in blocks])
spread = gblocks.std(axis=0,ddof=1)
fig, axes = plt.subplots(1,2,figsize=(12,4))
axes[0].plot(r,g['OO'],label='OO: minimum image',c='#2563eb')
axes[0].fill_between(r,np.maximum(0,g['OO']-spread),g['OO']+spread,alpha=.2,color='#2563eb',label='± block SD')
axes[0].plot(r,g['OO_no_PBC'],'--',label='OO: without PBC (wrong)',c='#dc2626')
for name in ('OO','OH','HH'): axes[1].plot(r,g[name],label=name)
for ax in axes:
    ax.axhline(1,c='gray',ls=':'); ax.set(xlabel='r / nm',ylabel='g(r)',xlim=(0,L/2)); ax.legend()
axes[0].set_title('Boundary correction is essential')
axes[1].set_title('Intermolecular structure of TIP3P water')
plt.tight_layout(); plt.show()
print('Количество RDF-кадров:',len(rdf_frames),'; ширина бина:',edges[1]-edges[0],'нм')
print('Допустимые пары:',npairs)
assert len(io)==N*(N-1)//2
assert len(aoh)==len(ih)==2*N*(N-1)

### 9. Как читать RDF и посчитать координационное число

У $g_{OO}$ первая выраженная оболочка ожидается примерно около 0.28 нм, между первой и второй оболочками обычно виден минимум около 0.33–0.35 нм. Точные положения и высоты зависят от модели, плотности, температуры и статистики: ниже они измеряются по нашей траектории, а не задаются как обязательный результат.

Межмолекулярный пик $g_{OH}$ на меньших расстояниях отражает близость кислорода одной воды и водорода другой; это связано с водородными связями. Одного расстояния недостаточно для строгого подсчёта водородных связей: требуется также угловой критерий.

Координационное число — среднее количество других O ближе $r_c$ к выбранному O. Для принятой конечномерной нормировки:

$$
n_{OO}(r_c)=\frac{N-1}{V}\int_0^{r_c}4\pi r^2g_{OO}(r)\,dr
=\frac{2}{MN}\sum_{r_{k+1}\le r_c}H_k.
$$

Выберем учебный порог 0.34 нм и округлим его к ближайшей границе бина. Это не автоматическое точное определение минимума. Сравним интеграл RDF с прямым подсчётом соседей.

In [ ]:
rc_target = .34
edge_index = np.argmin(np.abs(edges-rc_target))
rc = edges[edge_index]
coord_curve = np.cumsum(g['OO']*shell*(N-1)/V)
coord_rdf = coord_curve[edge_index-1]
coord_direct = 2*hist['OO'][:,:edge_index].sum()/(len(rdf_frames)*N)
peak_mask = (r>.24)&(r<.32)
peak = r[peak_mask][np.argmax(g['OO'][peak_mask])]
print(f'Первый OO-пик в окне 0.24–0.32 нм: {peak:.3f} нм')
print(f'Число O-соседей до {rc:.3f} нм: RDF = {coord_rdf:.3f}; прямой счёт = {coord_direct:.3f}')
assert np.isclose(coord_rdf,coord_direct)
fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot(edges[1:],coord_curve); ax.axvline(rc,ls='--',c='gray')
ax.scatter([rc],[coord_rdf],c='#dc2626')
ax.set(xlabel='r / nm',ylabel='Mean number of O neighbours',title='OO coordination number')
plt.show()

# RDF должно быть инвариантно к общему сдвигу начала координат.
p0 = wrapped[0]
p1 = np.mod(p0 + np.array([.37,-.22,.41])*L,L)
d0 = np.linalg.norm(mic(p0[O[jo]]-p0[O[io]]),axis=1)
d1 = np.linalg.norm(mic(p1[O[jo]]-p1[O[io]]),axis=1)
assert np.allclose(d0,d1,atol=1e-10)
print('Инвариантность периодических парных расстояний к сдвигу: OK')

### Задания

1. Пересчитайте MSD после уменьшения частоты записи. На каком интервале unwrapping начинает расходиться с полной исходной траекторией? Сравните с условием $|\Delta x|<L/2$ и помните, что редкая траектория сама по себе не сообщает число пропущенных обходов.
2. Сдвиньте начало ячейки на произвольный вектор. Изменятся ли wrapped-MSD, правильный MSD и RDF? Объясните различия.
3. Постройте OH-RDF с включёнными внутримолекулярными парами. Где появится пик и почему он не является пиком межмолекулярной водородной связи? Для нового набора пар измените нормировку.
4. Увеличьте $L$ до 2.4 нм и повторите расчёт. Сравните плотность, RDF и MSD; оставьте одинаковыми температуру, шаг и cutoff. Какие различия можно объяснить статистикой, а какие требуют исследования конечного размера?
5. Сравните MSD кислородов и центров масс **целых молекул**. Почему на коротких временах они могут отличаться из-за вращения, а в диффузионном пределе описывают один перенос молекулы?

### Документация и источники параметров

- [OpenMM: построение системы, Modeller.addSolvent](https://docs.openmm.org/8.0.0/userguide/application/03_model_building_editing.html).
- [OpenMM: запуск расчёта, ограничения и периодические взаимодействия](https://docs.openmm.org/8.0.0/userguide/application/02_running_sims.html).
- [OpenMM Context.getState: смысл enforcePeriodicBox](https://docs.openmm.org/latest/api-python/generated/openmm.openmm.Context.html).
- Числа для TIP3P проверены по `openmm/app/data/tip3p.xml` установленной версии 8.3.1 и напечатаны из созданного System. Время и численные результаты могут немного отличаться между CPU и GPU даже с одинаковым seed.

**Диффузия:** разделы 7A–7D сопоставляют Эйнштейна в NVT и Грина–Кубо в NVE. Дополнительный MSD NVE проверяет связь формул на одной динамике; дрейф энергии и зависимость интеграла VACF от предела помогают оценить надёжность.